In [4]:
# ============================================================
# FINAL MULTI-CANCER MULTI-OMICS PIPELINE
# Aggregated Analysis + Publication Quality Plots
# ============================================================
!pip install captum
import os
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import shap 

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE
from collections import defaultdict

# ============================================================
# CONFIG & STYLE
# ============================================================

SEED = 42
MAX_EPOCHS = 500
MIN_EPOCHS = 100
PATIENCE = 45
LR = 1e-3
WEIGHT_DECAY = 5e-4

# Logic & Loss Weights
ALIGN_W = 0.2    
ORTHO_W = 0.1       
GATE_ENT_W = 0.05   
SPARSITY_W = 0.01   

OMICS_DROPOUT_P = 0.15
TOP_K_GENES = 20
DEFAULT_N_SPLITS = 5
LATENT_DIM = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)

BASE_DIR = "./preprocessing/processed_multicancer"
RESULTS_ROOT = "./results_aggregated"
os.makedirs(RESULTS_ROOT, exist_ok=True)

GS_TYPES = ["GS-BRCA", "GS-LGG", "GS-OV", "GS-COAD", "GS-GBM"]
OMICS = ["mRNA", "miRNA", "CNV", "Methy"]

# Set Publication Quality Plotting Style
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# ============================================================
# 1. LOSS FUNCTIONS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.smoothing = smoothing

    def forward(self, logits, targets):
        n = logits.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(logits)
            true_dist.fill_(self.smoothing / (n - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)
        focal = (1 - p) ** self.gamma
        loss = -true_dist * focal * log_p
        if self.alpha is not None:
            loss = loss * self.alpha
        return loss.sum(dim=1).mean()

def alignment_loss(zs):
    loss = 0
    for i in range(len(zs)):
        for j in range(i + 1, len(zs)):
            zi = F.normalize(zs[i], dim=1)
            zj = F.normalize(zs[j], dim=1)
            loss += 1 - (zi * zj).sum(dim=1).mean()
    return loss

def orthogonality_loss(zs):
    loss = 0
    for i in range(len(zs)):
        for j in range(i + 1, len(zs)):
            zi = F.normalize(zs[i], dim=1)
            zj = F.normalize(zs[j], dim=1)
            loss += torch.abs((zi * zj).sum(dim=1)).mean()
    return loss

def gate_entropy(g):
    eps = 1e-6
    return -(g * torch.log(g + eps) + (1 - g) * torch.log(1 - g + eps)).mean()

# ============================================================
# 2. MODEL ARCHITECTURE
# ============================================================

class OmicsEncoder(nn.Module):
    def __init__(self, in_dim, latent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

class ContextGate(nn.Module):
    def __init__(self, context_dim, gate_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(context_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, gate_dim),
            nn.Sigmoid()
        )

    def forward(self, context):
        return self.fc(context)

class GatedMultiOmicsClassifier(nn.Module):
    def __init__(self, in_dims, num_classes):
        super().__init__()
        self.encoders = nn.ModuleDict({k: OmicsEncoder(v, LATENT_DIM) for k, v in in_dims.items()})
        total_latent_dim = LATENT_DIM * len(in_dims)
        self.gates = nn.ModuleDict({k: ContextGate(total_latent_dim, LATENT_DIM) for k in in_dims})
        
        self.classifier = nn.Sequential(
            nn.Linear(total_latent_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_dict):
        zs_map = {m: self.encoders[m](x_dict[m]) for m in self.encoders}
        all_z_list = [zs_map[m] for m in self.encoders]
        global_context = torch.cat(all_z_list, dim=1) 
        
        zs_list = []
        gated_list = []
        gates_map = {}

        for m in self.encoders:
            z = zs_map[m]
            g = self.gates[m](global_context)
            gated_z = z * g
            zs_list.append(z)
            gated_list.append(gated_z)
            gates_map[m] = g

        fused = torch.cat(gated_list, dim=1)
        logits = self.classifier(fused)
        
        return logits, zs_list, gates_map

# ============================================================
# 3. TRAINING & COLLECTION LOGIC
# ============================================================

def train_and_collect(omics_data, y, tr, te, cancer_name, fold_idx, accumulators):
    # Data Prep
    Xtr, Xte = {}, {}
    for m, X in omics_data.items():
        sc = StandardScaler()
        Xtr[m] = sc.fit_transform(X[tr])
        Xte[m] = sc.transform(X[te])

    ytr, yte = y[tr], y[te]
    Xtr_t = {m: torch.tensor(v, dtype=torch.float32).to(DEVICE) for m, v in Xtr.items()}
    Xte_t = {m: torch.tensor(v, dtype=torch.float32).to(DEVICE) for m, v in Xte.items()}
    ytr_t = torch.tensor(ytr, dtype=torch.long).to(DEVICE)

    w = compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)
    w = torch.tensor(w, dtype=torch.float32).to(DEVICE)
    loss_fn = FocalLoss(alpha=w)
    
    in_dims = {m: Xtr[m].shape[1] for m in Xtr}
    model = GatedMultiOmicsClassifier(in_dims, len(np.unique(y))).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_prec, wait = 0, 0
    best_state = None

    # --- Training Loop ---
    for epoch in range(MAX_EPOCHS):
        model.train()
        opt.zero_grad()
        
        train_inputs = {k: v.clone() for k,v in Xtr_t.items()}
        if torch.rand(1).item() < OMICS_DROPOUT_P:
            drop_omics = np.random.choice(list(train_inputs.keys()))
            train_inputs[drop_omics] = torch.zeros_like(train_inputs[drop_omics])

        logits, zs, gates = model(train_inputs)
        
        l_focal = loss_fn(logits, ytr_t) 
        l_align = ALIGN_W * alignment_loss(zs)
        l_ortho = ORTHO_W * orthogonality_loss(zs)
        l_ent = GATE_ENT_W * sum(gate_entropy(g) for g in gates.values())
        l_sparse = SPARSITY_W * sum(g.mean() for g in gates.values())

        loss = l_focal + l_align + l_ortho + l_ent + l_sparse
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            preds = model(Xte_t)[0].argmax(1).cpu().numpy()
            prec = precision_score(yte, preds, average="weighted", zero_division=0)

        if prec > best_prec:
            best_prec = prec
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if epoch >= MIN_EPOCHS and wait >= PATIENCE:
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    # ========================================================
    # DATA COLLECTION (No Plotting Yet)
    # ========================================================
    
    # 1. Collect Predictions for Confusion Matrix
    model.eval()
    with torch.no_grad():
        preds = model(Xte_t)[0].argmax(1).cpu().numpy()
        accumulators['y_true'].extend(yte)
        accumulators['y_pred'].extend(preds)
        
        # 2. Collect Gate Values
        _, _, gate_vals = model(Xte_t)
        for m in OMICS:
            # Store mean gate value for this batch
            accumulators['gates'][m].append(gate_vals[m].mean().item())

    # 3. Collect Gene Sensitivity
    for m in OMICS:
        inputs = Xte_t[m].clone().requires_grad_(True)
        full_inputs = {k: Xte_t[k] for k in OMICS}
        full_inputs[m] = inputs 
        
        logits, _, _ = model(full_inputs)
        score = logits.max(dim=1)[0].mean()
        score.backward()
        
        sens = inputs.grad.abs().mean(dim=0).cpu().numpy()
        
        if accumulators['sensitivity'][m] is None:
            accumulators['sensitivity'][m] = np.zeros_like(sens)
        
        # Accumulate sums (we will average later)
        if accumulators['sensitivity'][m].shape == sens.shape:
             accumulators['sensitivity'][m] += sens

    return best_prec

# ============================================================
# 4. FINAL AGGREGATED VISUALIZATION
# ============================================================

def generate_aggregated_plots(cancer_name, accumulators, n_folds):
    save_dir = os.path.join(RESULTS_ROOT, cancer_name)
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. Gate Importance (Bar Plot with Error Bars if we had them, here just Mean)
    # We have a list of means (one per fold). We can show the average across folds.
    gate_means = {m: np.mean(vals) for m, vals in accumulators['gates'].items()}
    gate_stds = {m: np.std(vals) for m, vals in accumulators['gates'].items()}
    
    plt.figure(figsize=(6, 5))
    x_vals = list(gate_means.keys())
    y_vals = list(gate_means.values())
    y_err = list(gate_stds.values())
    
    # Paper quality bar plot
    sns.barplot(x=x_vals, y=y_vals, palette="viridis", capsize=0.1)
    plt.errorbar(x=x_vals, y=y_vals, yerr=y_err, fmt='none', c='black', capsize=5)
    plt.title(f"{cancer_name}: Aggregated Gate Importance", fontweight='bold')
    plt.ylabel("Mean Gate Activation (0-1)")
    plt.xlabel("Omic Modality")
    plt.ylim(0, 1.1)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "aggregated_gate_importance.png"))
    plt.close()

    # 2. Confusion Matrix (Aggregated)
    cm = confusion_matrix(accumulators['y_true'], accumulators['y_pred'])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={"size": 14})
    plt.title(f"{cancer_name}: Aggregated Confusion Matrix", fontweight='bold')
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "aggregated_confusion_matrix.png"))
    plt.close()

    # 3. Top 20 Genes (Averaged across folds)
    for m in OMICS:
        if accumulators['sensitivity'][m] is not None:
            # Average the accumulated sensitivity
            avg_sens = accumulators['sensitivity'][m] / n_folds
            
            # Find Top 20
            idx = np.argsort(avg_sens)[::-1][:TOP_K_GENES]
            top_genes = avg_sens[idx]
            
            # Plot
            plt.figure(figsize=(8, 6))
            sns.barplot(x=top_genes, y=[f"Feat_{i}" for i in idx], palette="magma", orient='h')
            plt.title(f"{cancer_name} ({m}): Top 20 Features (Avg)", fontweight='bold')
            plt.xlabel("Sensitivity Score")
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"aggregated_top20_{m}.png"))
            plt.close()
            
            # Save CSV
            df = pd.DataFrame({"Feature_Index": idx, "Avg_Sensitivity": top_genes})
            df.to_csv(os.path.join(save_dir, f"aggregated_top20_{m}.csv"), index=False)

# ============================================================
# 5. MAIN LOOP
# ============================================================

if __name__ == "__main__":
    for CANCER in GS_TYPES:
        print(f"\n==================== {CANCER} ====================")
        DATA_DIR = os.path.join(BASE_DIR, CANCER)
        
        try:
            omics_data = {
                "mRNA":  np.load(os.path.join(DATA_DIR, "mRNA_processed.npy")),
                "miRNA": np.load(os.path.join(DATA_DIR, "miRNA_processed.npy")),
                "CNV":   np.load(os.path.join(DATA_DIR, "CNV_processed.npy")),
                "Methy": np.load(os.path.join(DATA_DIR, "Methy_processed.npy")),
            }
            y = np.load(os.path.join(DATA_DIR, "labels.npy"))
        except FileNotFoundError:
            print(f"Skipping {CANCER} (Files not found)")
            continue

        # Initialize Accumulators for this Cancer Type
        accumulators = {
            'y_true': [],
            'y_pred': [],
            'gates': defaultdict(list),
            'sensitivity': {m: None for m in OMICS}
        }

        # Dynamic Splits
        min_class_size = np.min(np.unique(y, return_counts=True)[1])
        n_splits = max(2, min_class_size) if min_class_size < DEFAULT_N_SPLITS else DEFAULT_N_SPLITS
        
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        fold_scores = []

        for i, (tr, te) in enumerate(skf.split(omics_data["mRNA"], y)):
            print(f"  > Fold {i}...")
            prec = train_and_collect(omics_data, y, tr, te, CANCER, i, accumulators)
            fold_scores.append(prec)
            plt.close('all')

        print(f"{CANCER} Mean Precision: {np.mean(fold_scores):.4f}")
        
        # Generate FINAL AGGREGATED PLOTS
        print(f"  > Generating aggregated plots for {CANCER}...")
        generate_aggregated_plots(CANCER, accumulators, n_splits)

    print("\nALL PROCESSES COMPLETED.")



==================== GS-BRCA ====================
  > Fold 0...
  > Fold 1...
  > Fold 2...
  > Fold 3...
  > Fold 4...
GS-BRCA Mean Precision: 0.8835
  > Generating aggregated plots for GS-BRCA...


/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:310: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_vals, y=y_vals, palette="viridis", capsize=0.1)
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i in idx], palette="magma", orient='h')
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i


==================== GS-LGG ====================
  > Fold 0...
  > Fold 1...
  > Fold 2...
  > Fold 3...
  > Fold 4...
GS-LGG Mean Precision: 0.9847
  > Generating aggregated plots for GS-LGG...


/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:310: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_vals, y=y_vals, palette="viridis", capsize=0.1)
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i in idx], palette="magma", orient='h')
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i


==================== GS-OV ====================
  > Fold 0...
  > Fold 1...
  > Fold 2...
  > Fold 3...
  > Fold 4...
GS-OV Mean Precision: 0.8899
  > Generating aggregated plots for GS-OV...


...

/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:310: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_vals, y=y_vals, palette="viridis", capsize=0.1)
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i in idx], palette="magma", orient='h')
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i


==================== GS-GBM ====================
  > Fold 0...
  > Fold 1...
  > Fold 2...
  > Fold 3...
  > Fold 4...
GS-GBM Mean Precision: 0.8395
  > Generating aggregated plots for GS-GBM...


/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:310: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=x_vals, y=y_vals, palette="viridis", capsize=0.1)
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i in idx], palette="magma", orient='h')
/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i


ALL PROCESSES COMPLETED.


/var/folders/9r/fjxy7zrn0jj7zcjz76fpx4mr0000gn/T/ipykernel_57416/1460746732.py:343: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_genes, y=[f"Feat_{i}" for i in idx], palette="magma", orient='h')
